### Tasks & Cancellation

In [ ]:
import asyncio
import time

start_time = time.monotonic()

def log(message: str) -> None:
    elapsed = time.monotonic() - start_time
    print(f"[{elapsed:6.2f}s] {message}")

In [ ]:
async def make_item(item: str = "coffee", brew: float = 2.0, fail: bool = False) -> str:
    try:
        await asyncio.sleep(brew)
    except asyncio.CancelledError:
        log(f"{item}: thrown out")
        raise
    if fail:
        raise RuntimeError(f"out of {item}")
    return item

**Start a task, check status**

In [ ]:
start_time = time.monotonic()

task = asyncio.create_task(make_item("espresso", brew=1.5))
log(f"done() right after create: {task.done()}")

cup = await task  # the task actually starts running here — at the next await
log(f"done() after await: {task.done()}")
log(f"result: {cup}")

**Cancel after a timeout** — `make_item` cleans up in its `except` and re-raises

In [ ]:
start_time = time.monotonic()

cup = asyncio.create_task(make_item("slow drip", brew=5.0))
await asyncio.sleep(1.5)
cup.cancel(msg="café closing")

try:
    await cup
except asyncio.CancelledError as error:
    log(f"caught CancelledError: {error}")

log(f"cancelled() -> {cup.cancelled()}")

**Cancel many at once**

In [ ]:
start_time = time.monotonic()

cups = [asyncio.create_task(make_item(f"cup-{index}", brew=5.0)) for index in range(3)]
await asyncio.sleep(1.0)

for cup in cups:
    cup.cancel()

results = await asyncio.gather(*cups, return_exceptions=True)
for cup, outcome in zip(cups, results):
    log(f"{cup.get_name()}: outcome={type(outcome).__name__}")

---
- `create_task()` starts background work; `cancel()` stops it via `CancelledError`.
- `make_item` re-raises `CancelledError` after cleanup so cancellation propagates.
- `gather(..., return_exceptions=True)` collects errors instead of re-raising.